In [1]:
%matplotlib tk
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# importing sample data
X = np.array(([3, 5], [5, 1], [10, 2]), dtype = float)
y = np.array(([75], [82], [93]), dtype = float)

In [3]:
X = X / np.amax(X, axis = 0)
y = y / 100 # 100 is the max score  

In [4]:
X

array([[0.3, 1. ],
       [0.5, 0.2],
       [1. , 0.4]])

In [5]:
y

array([[0.75],
       [0.82],
       [0.93]])

In [6]:
class NeuralNetwork(object):
    def __init__(self):
        # hyperparameters
        self.inputLayerSize = 2
        self.hiddenLayerSize = 3
        self.outputLayerSize = 1
        
        # weights
        self.W1 = np.random.randn(self.inputLayerSize, self.hiddenLayerSize)
        self.W2 = np.random.randn(self.hiddenLayerSize, self.outputLayerSize)
        
    def forward(self, X):
        self.z2 = np.dot(X, self.W1)
        self.a2 = self.sigmoid(self.z2)
        self.z3 = np.dot(self.a2, self.W2)
        yHat = self.sigmoid(self.z3)
        return yHat 
    
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))
    
    def d_sigmoid(self, z):
        return np.exp(-z) / ((1 + np.exp(-z)) ** 2)
    
    def costFunction(self, X, y):
        yHat = self.forward(X)
        return 0.5 * np.sum((y - yHat) ** 2)
    
    def costFunctionPrime(self, X, y):
        self.yHat = self.forward(X)
        
        delta3 = np.multiply(-(y - self.yHat), self.d_sigmoid(self.z3))
        dJdW2 = np.dot(self.a2.T, delta3)
        
        delta2 = np.dot(delta3, self.W2.T) * self.d_sigmoid(self.z2)
        dJdW1 = np.dot(X.T, delta2)
        
        return dJdW1, dJdW2
    
    #defining helper functions
    def getParams(self): #returning the weights
        params = np.hstack((self.W1.ravel(), self.W2.ravel()))
        return params
        
    def setParams(self, params):
        W1_start = 0
        W1_end = self.hiddenLayerSize * self.inputLayerSize
        self.W1 = np.reshape(params[W1_start:W1_end], (self.inputLayerSize, self.hiddenLayerSize))
        W2_end = W1_end + self.hiddenLayerSize * self.outputLayerSize
        self.W2 = np.reshape(params[W1_end: W2_end], (self.hiddenLayerSize, self.outputLayerSize))
        
    def computeGradient(self, X, y):
        dJ1W1, dJ2W2 = self.costFunctionPrime(X, y)
        return np.concatenate((dJ1W1.ravel(), dJ2W2.ravel()))

In [7]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [8]:
testInput = np.arange(-6, 6, .01)
plt.plot(testInput, sigmoid(testInput))
plt.grid(1)

In [9]:
sigmoid(np.random.randn(3, 3))

array([[0.46887297, 0.55482659, 0.82509943],
       [0.50121694, 0.35725697, 0.63575057],
       [0.25335257, 0.37618226, 0.52172212]])

In [10]:
NN = NeuralNetwork()
yHat = NN.forward(X)

In [11]:
yHat

array([[0.45616319],
       [0.41576568],
       [0.3862436 ]])

In [12]:
plt.bar([0, 1, 2], y.flatten(), width = 0.3)
plt.bar([0.3, 1.3, 2.3], yHat.flatten(), color = 'r', width = 0.3)
plt.grid(1)
plt.legend(['y', 'yHat'])

In [13]:
#implement a stochastic gradient descent

In [14]:
NN = NeuralNetwork()
cost1 = NN.costFunction(X, y)
dJ1dw1, dJ2dw2 = NN.costFunctionPrime(X, y)

In [15]:
scalar = 3
NN.W1 = NN.W1 + scalar * dJ1dw1
NN.W2 = NN.W2 + scalar * dJ2dw2
cost2 = NN.costFunction(X, y)
print(cost1, cost2)

0.06502081387559525 0.10759593290648722


In [16]:
dJ1dw1, dJ2dw2 = NN.costFunctionPrime(X, y)

In [17]:
NN.W1 = NN.W1 - scalar * dJ1dw1
NN.W2 = NN.W2 - scalar * dJ2dw2
cost3 = NN.costFunction(X, y)
print(cost2, cost3)

0.10759593290648722 0.05224762881829954


In [18]:
a = np.array([[1, 2, 3, 4]])
b = np.array([[5, 6, 7, 8]])
# print(a.shape, b.shape)
# print(np.concatenate([a, b]), np.concatenate([a, b]).shape)
# print(np.concatenate((a, b)))
# c = np.concatenate([a, b.T], axis=1)
# print(c)
# d = np.hstack((a, b.T))
# print(d)
# concatenate, hstack, vstack, stack
e = np.hstack((a, b), dtype = float)
print(e, e.shape)

[[1. 2. 3. 4. 5. 6. 7. 8.]] (1, 8)


In [19]:
def computeGradients(N, X, y): #numerical gradient checking between the proper differtial methods and derivative equation 
    e = 1e-4
    paramsInitial = N.getParams()
    numgrad = np.zeros(paramsInitial.shape)
    perturb = np.zeros(paramsInitial.shape)
    
    for p in range(len(paramsInitial)):
        perturb[p] = e
        N.setParams(paramsInitial + perturb)
        loss2 = N.costFunction(X, y)
        N.setParams(paramsInitial - perturb)
        loss1 = N.costFunction(X, y)
        
        numgrad[p] = (loss2 - loss1) / (2 * e)
        perturb[p] = 0
    
    N.setParams(paramsInitial)
    return numgrad

In [20]:
N = NeuralNetwork()

In [21]:
numgrad = computeGradients(N, X, y)
numgrad

array([ 0.04612782, -0.03787248, -0.01201562,  0.02136504, -0.01702296,
       -0.0062195 , -0.07613676, -0.07575033, -0.11662402])

In [22]:
grad = N.computeGradient(X, y)
grad

array([ 0.04612782, -0.03787248, -0.01201562,  0.02136504, -0.01702296,
       -0.0062195 , -0.07613676, -0.07575033, -0.11662402])

In [23]:
np.linalg.norm(grad - numgrad)/np.linalg.norm(grad + numgrad)

np.float64(1.7351313514082366e-10)

In [24]:
from scipy import optimize #for running BFGS algorithm for minimization

In [25]:
class trainer(object):
    def __init__(self, N):
        self.N = N
        
    def callbackF(self, params):
        self.N.setParams(params)
        self.J.append(self.N.costFunction(self.X, self.y))
        
    def costFunctionWrapper(self, params, X, y):
        self.N.setParams(params)
        cost = self.N.costFunction(X, y)
        grad = self.N.computeGradient(X, y)
        return cost, grad
        
    def train(self, X, y):
        self.X = X
        self.y = y
        
        self.J = []
        
        params0= self.N.getParams() 
        
        options = {'maxiter': 200, 'disp': True}
        _res = optimize.minimize(self.costFunctionWrapper, params0, jac=True, method = 'BFGS', args=(X, y), options = options, callback = self.callbackF)
        
        self.N.setParams(_res.x)
        self.optimizeResults = _res

In [26]:
NN = NeuralNetwork()

In [27]:
T = trainer(NN)

In [28]:
T.train(X, y)

Optimization terminated successfully.
         Current function value: 0.000000
         Iterations: 59
         Function evaluations: 61
         Gradient evaluations: 61


In [29]:
plt.plot(T.J)
plt.grid(1)
plt.xlabel('cost')
plt.ylabel('Iterations')

Text(42.722222222222214, 0.5, 'Iterations')

In [30]:
NN.costFunctionPrime(X, y)

(array([[ 3.45244121e-06,  2.93721360e-06, -8.24491463e-06],
        [ 1.52832541e-06,  1.46015333e-06, -3.64196141e-06]]),
 array([[6.15127014e-06],
        [8.90463362e-06],
        [4.66547848e-06]]))

In [31]:
NN.forward(X)

array([[0.75000208],
       [0.8200736 ],
       [0.92999915]])

In [32]:
y

array([[0.75],
       [0.82],
       [0.93]])

In [33]:
hoursSlept = np.linspace(0, 10, 100)
hoursStudied = np.linspace(0, 5, 100)

hoursSleptNorm = hoursSlept/10.
hoursStudiedNorm = hoursStudied/5.

a, b = np.meshgrid(hoursSleptNorm, hoursStudiedNorm)

allInputs = np.zeros((a.size, 2))
allInputs[:, 0] = a.ravel()
allInputs[:, 1] = b.ravel()

In [34]:
allOutputs = NN.forward(allInputs)

In [42]:
yy = np.dot(hoursStudied.reshape(100, 1), np.ones((1, 100)))
xx = np.dot(hoursSlept.reshape(100, 1), np.ones((1, 100))).T

from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
fig = plt.figure()
ax = fig.add_subplot(projection = '3d')

ax.set_zlim(0, 100)


surf = ax.plot_surface(xx, yy, 100 * allOutputs.reshape(100, 100), cmap = cm.jet)
ax.set_xlabel('hours slept')
ax.set_ylabel('hours studied')
ax.set_zlabel('marks')

plt.show()